In [1]:
import math
data = [
    ("Sunny",    "Hot",  "High",   "Weak",   "No"),
    ("Sunny",    "Hot",  "High",   "Strong", "No"),
    ("Overcast", "Hot",  "High",   "Weak",   "Yes"),
    ("Rain",     "Mild", "High",   "Weak",   "Yes"),
    ("Rain",     "Cool", "Normal", "Weak",   "Yes"),
    ("Rain",     "Cool", "Normal", "Strong", "No"),
    ("Overcast", "Cool", "Normal", "Strong", "Yes"),
    ("Sunny",    "Mild", "High",   "Weak",   "No"),
    ("Sunny",    "Cool", "Normal", "Weak",   "Yes"),
    ("Rain",     "Mild", "Normal", "Weak",   "Yes"),
    ("Sunny",    "Mild", "Normal", "Strong", "Yes"),
    ("Overcast", "Mild", "High",   "Strong", "Yes"),
    ("Overcast", "Hot",  "Normal", "Weak",   "Yes"),
    ("Rain",     "Mild", "High",   "Strong", "No"),
]

def entropie(dataset):
    h, nbr_yes, nbr_no = 0, 0, 0
    for i in dataset:
        if i[4] == 'Yes':
            nbr_yes = nbr_yes + 1
        else:
            nbr_no = nbr_no + 1
    if nbr_yes == 0 or nbr_no == 0:
        return 0
    p_yes = nbr_yes / len(dataset)
    p_no  = nbr_no  / len(dataset)
    h = -(p_yes * math.log2(p_yes)) - (p_no * math.log2(p_no))
    return h

def entropie_attribut(dataset,colon):
    totale=len(dataset)
    h_attribut = 0
    valeurs=set(row[colon] for row in dataset) # Étape 1 : quelles valeurs existe dans cette colonne ?
    for i in valeurs:
        subset=[row for row in dataset if row[colon]==i] # Étape 2 : garder seulement les lignes où col == i
        poids=len(subset)/totale # Étape 3 : calculer le poids de ce groupe
        h_attribut += poids * entropie(subset)  # Étape 4 : ajouter la contribution de ce groupe
    return h_attribut

def gain(dataset, col):
    return entropie(dataset) - entropie_attribut(dataset, col)

def meilleur_attribut(dataset, attributs):
    meilleur_gain=-1
    meilleur_attr=None
    for col in attributs:
        g = gain(dataset, col)
        print(f"  Gain(col={col}) = {g:.4f}")  # pour visualiser
        if g > meilleur_gain:
            meilleur_gain = g
            meilleur_attr = col

    return meilleur_attr
def id3(dataset,attributs):
   # ── Cas 1 : tout Yes ?
    if all(row[4] == "Yes" for row in dataset):
        return "Yes"
   # ── Cas 1 : tout Non ?
    if all(row[4] == "Non" for row in dataset):
        return "Non"
     # ── Cas 3 : plus d'attributs → classe majoritaire
    if len(attributs) == 0:
        nbr_yes = sum(1 for row in dataset if row[4] == "Yes")
        nbr_no  = len(dataset) - nbr_yes
        return "Yes" if nbr_yes >= nbr_no else "No"
   # ── Cas 4 : choisir le meilleur attribut
    best_col = meilleur_attribut(dataset, attributs)
    # Créer le noeud de l'arbre
    arbre = {best_col: {}}

    # Retirer best_col des attributs restants
    attributs_restants = [a for a in attributs if a != best_col]

    # Pour chaque valeur possible de best_col
    valeurs = set(row[best_col] for row in dataset)
    for v in valeurs:
        subset = [row for row in dataset if row[best_col] == v]

        # Appel récursif
        arbre[best_col][v] = id3(subset, attributs_restants)

    return arbre

print(entropie(data))
print(entropie_attribut(data, 1))
attributs = [0, 1, 2, 3]  # Outlook, Temperature, Humidity, Wind
best = meilleur_attribut(data, attributs)
print(f"\nMeilleur attribut : colonne {best}")
import json

attributs = [0, 1, 2, 3]
arbre = id3(data, attributs)

print(json.dumps(arbre, indent=4))

0.9402859586706311
0.9110633930116763
  Gain(col=0) = 0.2467
  Gain(col=1) = 0.0292
  Gain(col=2) = 0.1518
  Gain(col=3) = 0.0481

Meilleur attribut : colonne 0
  Gain(col=0) = 0.2467
  Gain(col=1) = 0.0292
  Gain(col=2) = 0.1518
  Gain(col=3) = 0.0481
  Gain(col=1) = 0.0200
  Gain(col=2) = 0.0200
  Gain(col=3) = 0.9710
  Gain(col=1) = 0.0000
  Gain(col=2) = 0.0000
  Gain(col=2) = 0.0000
  Gain(col=2) = 0.0000
  Gain(col=1) = 0.5710
  Gain(col=2) = 0.9710
  Gain(col=3) = 0.0200
  Gain(col=1) = 0.0000
  Gain(col=3) = 0.0000
  Gain(col=3) = 0.0000
  Gain(col=3) = 0.0000
{
    "0": {
        "Overcast": "Yes",
        "Rain": {
            "3": {
                "Strong": {
                    "1": {
                        "Mild": {
                            "2": {
                                "High": "No"
                            }
                        },
                        "Cool": {
                            "2": {
                                "Normal": "No"
      